# Обучение базовых моделей

In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt, ticker as mticker
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.precision', 4)

In [2]:
from pathlib import Path

path_p1_p2 = Path("..", "data", "processed", "p1_p2_data.csv")
df_p1_p2 = pd.read_csv(path_p1_p2, sep=";")

path_p3 = Path("..", "data", "processed", "p3_data.csv")
df_p3 = pd.read_csv(path_p3, sep=";")

path_altpay5 = Path("..", "data", "processed", "altpay5_data.csv")
df_altpay5 = pd.read_csv(path_altpay5, sep=";")

dfs = [df_p1_p2, df_p3, df_altpay5]

COMMON_DTTM_COLS = [
    'month',
    'first_trx_month_inn',
    'last_active_month_inn',
]

CAT_COLS = [
    'inn_status',
    'top_mcc_group_inn',
    'who_is_this_first_inn',
]

for df in dfs:
    for col in COMMON_DTTM_COLS:
        df[col] = pd.to_datetime(df[col])

    # В каждом df разный набор колонок с датами подключения других продуктов
    first_month_cols = [col for col in df.columns if col.startswith("first_month")]
    for col in first_month_cols:
        df[col] = pd.to_datetime(df[col])

    for col in CAT_COLS:
        df[col] = df[col].astype('category')

## Метрики качества
Основной метрикой качества моделей является Precision@K, поскольку бизнес-задача заключается в эффективном распределении ограниченного ресурса отдела продаж. В рамках задачи capacity отдела составляет 2000 клиентов в месяц, что определяет основное значение K. Дополнительно рассматриваются значения K=500, K=1000 и K=1500 для анализа устойчивости качества ранжирования.

Для оценки относительного качества моделей используется метрика Uplift — отношение Precision@K рассматриваемой модели к Precision@K базовых подходов. В качестве базовых подходов используются:

1. Случайный отбор клиентов
2. Отбор на основе текущих бизнес-правил

Дополнительно для оценки качества ранжирования анализируется зависимость Precision@K от K, что позволяет оценить способность модели выделять наиболее релевантных клиентов в верхней части ранжированного списка.

Также используется метрика Recall@K, отражающая долю всех целевых событий (подключений продукта), попадающих в top-K клиентов, что позволяет оценить полноту охвата моделью потенциально релевантных клиентов.

Для интерпретации результатов в бизнес-терминах рассчитывается ожидаемое количество подключений (Expected Conversions), определяемое как произведение Precision@K на K.

## P1P2

### Случайный отбор


Для случайной модели ранжирования ожидаемое значение Precision@K совпадает с долей положительных объектов в выборке. В связи с этим в качестве baseline используется доля положительного класса, рассчитанная по каждому месяцу.

In [3]:
k_values = [500, 1000, 1500, 2000, 2500]

target_monthly = df_p1_p2.groupby('month').target.sum()
month_sizes = df_p1_p2.groupby('month').size()

precision = (target_monthly / month_sizes).mean()  # macro average

print("Random Selection P1P2")

for k in k_values:
    coverage = (k / month_sizes).mean()
    expected_conversions = precision * k

    print(
        f"K={k:<4} | "
        f"Precision: {precision:.2%} | "
        f"Coverage: {coverage:.2%} | "
        f"Expected Conversions: {expected_conversions:.2f}"
    )

Random Selection P1P2
K=500  | Precision: 0.23% | Coverage: 2.23% | Expected Conversions: 1.15
K=1000 | Precision: 0.23% | Coverage: 4.45% | Expected Conversions: 2.30
K=1500 | Precision: 0.23% | Coverage: 6.68% | Expected Conversions: 3.45
K=2000 | Precision: 0.23% | Coverage: 8.91% | Expected Conversions: 4.59
K=2500 | Precision: 0.23% | Coverage: 11.13% | Expected Conversions: 5.74


### Business-Rule

Критерии релевантности:
- Отсутствие подключённых продуктов `P1`, `P2` (фильтрация уже реализована в `df_p1_p2`)
- Статус `'Active'`, `'Reborn'`
- Группа MCC `!= 'Charity'`

Кандидаты на подключение ранжируется по убыванию `turnover`

In [4]:
def add_rule_flag_p1_p2(df: pd.DataFrame) -> pd.DataFrame:
    active_statuses = ["Active", "Reborn"]
    banned_mcc_groups = ["Charity"]

    df = df.copy()
    
    df["rule_flag"] = (
        df["inn_status"].isin(active_statuses)
        & ~df["top_mcc_group_inn"].isin(banned_mcc_groups)
    ).astype("int8")

    return df


df_p1_p2 = add_rule_flag_p1_p2(df_p1_p2)

Замерим качество.

In [5]:
k_values = [500, 1000, 1500, 2000, 2500]
y_score = 'turnover'

print(f"Business-Rule Selection P1P2, y_score='{y_score}'")

for k in k_values:
    df_k = (
        df_p1_p2.query('rule_flag == 1')
        .sort_values(by=y_score, ascending=False)
        .groupby('month')
        .head(k)
    )

    tp = df_k[df_k['target'] == 1].groupby('month').size()
    fp = df_k[df_k['target'] == 0].groupby('month').size()

    total_positives = (
        df_p1_p2[df_p1_p2['target'] == 1]
        .groupby('month')
        .size()
    )

    tp, fp = tp.align(fp, fill_value=0)
    tp, total_positives = tp.align(total_positives, fill_value=0)

    precision_k = (tp / (tp + fp)).mean()
    recall_k = (tp / total_positives).mean()

    avg_rows = df_k.groupby('month').size().mean()
    expected_conversions = precision_k * avg_rows

    print(
        f"K={k:<4} | "
        f"Precision: {precision_k:.2%} | "
        f"Recall: {recall_k:.2%} | "
        f"Expected Conversions: {expected_conversions:.2f}"
    )

Business-Rule Selection P1P2, y_score='turnover'
K=500  | Precision: 0.56% | Recall: 5.67% | Expected Conversions: 2.80
K=1000 | Precision: 0.55% | Recall: 11.33% | Expected Conversions: 5.53
K=1500 | Precision: 0.52% | Recall: 16.06% | Expected Conversions: 7.80
K=2000 | Precision: 0.53% | Recall: 21.24% | Expected Conversions: 10.60
K=2500 | Precision: 0.53% | Recall: 26.61% | Expected Conversions: 13.33


Попробуем улучшить отбор по бизнес-правилу за счёт изменения столбца ранжирования. Переберём все числовые признаки, кроме `target` и `rule_flag`.

In [6]:
numeric_features = [col for col in df_p1_p2.select_dtypes(include=['number']).columns if col not in ('target', 'rule_flag')]

k_values = [500, 1000, 1500, 2000, 2500]

print(f"Business-Rule Selection P1P2")

for asc in (True, False):
    for feature in numeric_features:
        y_score = feature
        print(f"y_score='{y_score}' {'ASC' if asc else 'DESC'}")
        for k in k_values:
            df_k = (
                df_p1_p2.query('rule_flag == 1')
                .sort_values(by=y_score, ascending=asc)
                .groupby('month')
                .head(k)
            )

            tp = df_k[df_k['target'] == 1].groupby('month').size()
            fp = df_k[df_k['target'] == 0].groupby('month').size()

            total_positives = (
                df_p1_p2[df_p1_p2['target'] == 1]
                .groupby('month')
                .size()
            )

            tp, fp = tp.align(fp, fill_value=0)
            tp, total_positives = tp.align(total_positives, fill_value=0)

            precision_k = (tp / (tp + fp)).mean()
            recall_k = (tp / total_positives).mean()

            avg_rows = df_k.groupby('month').size().mean()
            expected_conversions = precision_k * avg_rows

            print(
                f"K={k:<4} | "
                f"Precision: {precision_k:.2%} | "
                f"Recall: {recall_k:.2%} | "
                f"Expected Conversions: {expected_conversions:.2f}"
            )
        print()

Business-Rule Selection P1P2
y_score='lifetime_month_streak_inn' ASC
K=500  | Precision: 2.79% | Recall: 25.68% | Expected Conversions: 13.93
K=1000 | Precision: 1.84% | Recall: 34.44% | Expected Conversions: 18.40
K=1500 | Precision: 1.40% | Recall: 39.71% | Expected Conversions: 21.00
K=2000 | Precision: 1.13% | Recall: 43.03% | Expected Conversions: 22.67
K=2500 | Precision: 0.99% | Recall: 47.25% | Expected Conversions: 24.80

y_score='altpay1_turnover' ASC
K=500  | Precision: 0.72% | Recall: 6.81% | Expected Conversions: 3.60
K=1000 | Precision: 0.73% | Recall: 13.66% | Expected Conversions: 7.33
K=1500 | Precision: 0.73% | Recall: 20.60% | Expected Conversions: 10.93
K=2000 | Precision: 0.70% | Recall: 26.31% | Expected Conversions: 14.00
K=2500 | Precision: 0.73% | Recall: 34.69% | Expected Conversions: 18.27

y_score='altpay2_turnover' ASC
K=500  | Precision: 0.80% | Recall: 7.41% | Expected Conversions: 4.00
K=1000 | Precision: 0.80% | Recall: 15.01% | Expected Conversions: 8.

### Logistic Regression

Подготовим данные для обучения Логистической регрессии

In [7]:
df_p1_p2_preprocessed = df_p1_p2.copy()

df_p1_p2_preprocessed['months_since_last_active_months'] = (
    (df_p1_p2_preprocessed['month'].dt.year - df_p1_p2_preprocessed['last_active_month_inn'].dt.year) * 12 +
    (df_p1_p2_preprocessed['month'].dt.month - df_p1_p2_preprocessed['last_active_month_inn'].dt.month)
)

df_p1_p2_preprocessed["months_since_first_trx"] = (
    (df_p1_p2_preprocessed["month"].dt.year - df_p1_p2_preprocessed["first_trx_month_inn"].dt.year) * 12 +
    (df_p1_p2_preprocessed["month"].dt.month - df_p1_p2_preprocessed["first_trx_month_inn"].dt.month)
)

df_p1_p2_preprocessed["has_p3"] = df_p1_p2_preprocessed["first_month_p3"].notna()

df_p1_p2_preprocessed["has_altpay5"] = df_p1_p2_preprocessed["first_month_altpay5"].notna()

cols_to_drop = [
    'inn',
    "last_active_month_inn",
    "first_trx_month_inn",
    "top_mcc_group_inn",
    "who_is_this_first_inn",
    "first_month_p3",
    "first_month_altpay5",
]

df_p1_p2_preprocessed = (
    df_p1_p2_preprocessed
    .sort_values(by='month')
    .reset_index(drop=True)
    .drop(columns=cols_to_drop)
)

df_p1_p2_preprocessed = pd.get_dummies(df_p1_p2_preprocessed)

In [38]:
train = df_p1_p2_preprocessed.loc[df_p1_p2_preprocessed['month'] < '2025-12-01'] # 80%
test = df_p1_p2_preprocessed.loc[df_p1_p2_preprocessed['month'] >= '2025-12-01'] # 20%

X_train = train.drop(columns=['target', 'month'])
y_train = train['target']

X_test = test.drop(columns=['target', 'month'])
y_test = test['target']

In [39]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=100_000)

lr.fit(X_train, y_train)

test['lr_proba'] = lr.predict_proba(X_test)[:, 1]

In [40]:
pd.DataFrame(lr.coef_[0], index=lr.feature_names_in_).sort_values(by=0, ascending=False).head(10)

,0
is_first_trx_month,1.5367
is_relevant_mcc_altpay5,0.8984
is_relevant_turnover_altpay5,0.5304
rule_flag,0.3447
turnover_change_to_turnover_ma_6m,0.2610
altpay3_turnover,0.2549
cnt_trx,0.1824
cnt_trx_ma_6m,0.1747
cnt_trx_ma_2m,0.1641
cnt_trx_ma_3m,0.1632


In [41]:
pd.DataFrame(lr.coef_[0], index=lr.feature_names_in_).sort_values(by=0, ascending=True).head(10)

,0
inn_status_Churn,-2.4655
has_termination_start,-1.3104
is_left_time_border,-0.9293
avg_check_change_to_avg_check_wma_6m,-0.6079
turnover_cv_2m,-0.5172
inn_status_Active,-0.5165
inn_status_New Churn,-0.4832
turnover_ema_2m,-0.3905
turnover_ma_6m,-0.3815
turnover_ma_2m,-0.3613


In [42]:
k_values = [500, 1000, 1500, 2000, 2500]
y_score = 'lr_proba'

print(f"Logistic Regression P1P2, y_score='{y_score}'")

for k in k_values:
    df_k = (
        test
        .sort_values(by=y_score, ascending=False)
        .groupby('month')
        .head(k)
    )

    tp = df_k[df_k['target'] == 1].groupby('month').size()
    fp = df_k[df_k['target'] == 0].groupby('month').size()

    total_positives = (
        test[test['target'] == 1]
        .groupby('month')
        .size()
    )

    tp, fp = tp.align(fp, fill_value=0)
    tp, total_positives = tp.align(total_positives, fill_value=0)

    precision_k = (tp / (tp + fp)).mean()
    recall_k = (tp / total_positives).mean()

    avg_rows = df_k.groupby('month').size().mean()
    expected_conversions = precision_k * avg_rows

    print(
        f"K={k:<4} | "
        f"Precision: {precision_k:.2%} | "
        f"Recall: {recall_k:.2%} | "
        f"Expected Conversions: {expected_conversions:.2f}"
    )

Logistic Regression P1P2, y_score='lr_proba'
K=500  | Precision: 3.40% | Recall: 29.56% | Expected Conversions: 17.00
K=1000 | Precision: 2.17% | Recall: 37.72% | Expected Conversions: 21.67
K=1500 | Precision: 1.67% | Recall: 43.35% | Expected Conversions: 25.00
K=2000 | Precision: 1.42% | Recall: 49.32% | Expected Conversions: 28.33
K=2500 | Precision: 1.20% | Recall: 52.18% | Expected Conversions: 30.00


### LightGBM

In [ ]:
import lightgbm as lgb

### CatBoost

## P3

### Случайный отбор

In [33]:
k_values = [500, 1000, 1500, 2000, 2500]

target_monthly = df_p3.groupby('month').target.sum()
month_sizes = df_p3.groupby('month').size()

precision = (target_monthly / month_sizes).mean()  # macro average

print("Random Selection P1P2")

for k in k_values:
    coverage = (k / month_sizes).mean()
    expected_conversions = precision * k

    print(
        f"K={k:<4} | "
        f"Precision: {precision:.2%} | "
        f"Coverage: {coverage:.2%} | "
        f"Expected Conversions: {expected_conversions:.2f}"
    )

Random Selection P1P2
K=500  | Precision: 0.01% | Coverage: 1.94% | Expected Conversions: 0.03
K=1000 | Precision: 0.01% | Coverage: 3.87% | Expected Conversions: 0.07
K=1500 | Precision: 0.01% | Coverage: 5.81% | Expected Conversions: 0.10
K=2000 | Precision: 0.01% | Coverage: 7.74% | Expected Conversions: 0.14
K=2500 | Precision: 0.01% | Coverage: 9.68% | Expected Conversions: 0.17


### Business-Rule

Критерии релевантности:
- Отсутствие подключённого продукта `P3`
- Статус `'Active'`, `'Reborn'`
- Релевантные MCC (флаг `is_relevant_mcc_p3`)
- Группа MCC `!= 'Charity'`

In [34]:
def add_rule_flag_p3(df: pd.DataFrame) -> pd.DataFrame:
    active_statuses = ["Active", "Reborn"]
    banned_mcc_groups = ["Charity"]

    df = df.copy()
    
    df["rule_flag"] = (
        df["inn_status"].isin(active_statuses)
        & ~df["top_mcc_group_inn"].isin(banned_mcc_groups)
        & df["is_relevant_mcc_p3"]
    ).astype("int8")

    return df


df_p3 = add_rule_flag_p3(df_p3)

In [35]:
k_values = [500, 1000, 1500, 2000, 2500]
y_score = 'turnover'

print(f"Business-Rule Selection P3, y_score='{y_score}'")

for k in k_values:
    df_k = (
        df_p3.query('rule_flag == 1')
        .sort_values(by=y_score, ascending=False)
        .groupby('month')
        .head(k)
    )

    tp = df_k[df_k['target'] == 1].groupby('month').size()
    fp = df_k[df_k['target'] == 0].groupby('month').size()

    total_positives = (
        df_p3[df_p3['target'] == 1]
        .groupby('month')
        .size()
    )

    tp, fp = tp.align(fp, fill_value=0)
    tp, total_positives = tp.align(total_positives, fill_value=0)

    precision_k = (tp / (tp + fp)).mean()
    recall_k = (tp / total_positives).mean()

    avg_rows = df_k.groupby('month').size().mean()
    expected_conversions = precision_k * avg_rows

    print(
        f"K={k:<4} | "
        f"Precision: {precision_k:.2%} | "
        f"Recall: {recall_k:.2%} | "
        f"Expected Conversions: {expected_conversions:.2f}"
    )

Business-Rule Selection P3, y_score='turnover'
K=500  | Precision: 0.03% | Recall: 7.58% | Expected Conversions: 0.13
K=1000 | Precision: 0.02% | Recall: 10.61% | Expected Conversions: 0.20
K=1500 | Precision: 0.01% | Recall: 10.61% | Expected Conversions: 0.20
K=2000 | Precision: 0.02% | Recall: 28.79% | Expected Conversions: 0.47
K=2500 | Precision: 0.02% | Recall: 31.82% | Expected Conversions: 0.53


Попробуем также как с `P1P2` пересмотреть метрику ранжирования

In [36]:
numeric_features = [col for col in df_p3.select_dtypes(include=['number']).columns if col not in ('target', 'rule_flag')]

k_values = [500, 1000, 1500, 2000, 2500]

print(f"Business-Rule Selection P3")

for asc in (True, False):
    for feature in numeric_features:
        y_score = feature
        print(f"y_score='{y_score}' {'ASC' if asc else 'DESC'}")
        for k in k_values:
            df_k = (
                df_p3.query('rule_flag == 1')
                .sort_values(by=y_score, ascending=asc)
                .groupby('month')
                .head(k)
            )

            tp = df_k[df_k['target'] == 1].groupby('month').size()
            fp = df_k[df_k['target'] == 0].groupby('month').size()

            total_positives = (
                df_p3[df_p3['target'] == 1]
                .groupby('month')
                .size()
            )

            tp, fp = tp.align(fp, fill_value=0)
            tp, total_positives = tp.align(total_positives, fill_value=0)

            precision_k = (tp / (tp + fp)).mean()
            recall_k = (tp / total_positives).mean()

            avg_rows = df_k.groupby('month').size().mean()
            expected_conversions = precision_k * avg_rows

            print(
                f"K={k:<4} | "
                f"Precision: {precision_k:.2%} | "
                f"Recall: {recall_k:.2%} | "
                f"Expected Conversions: {expected_conversions:.2f}"
            )
        print()

Business-Rule Selection P3
y_score='lifetime_month_streak_inn' ASC
K=500  | Precision: 0.01% | Recall: 2.27% | Expected Conversions: 0.07
K=1000 | Precision: 0.01% | Recall: 2.27% | Expected Conversions: 0.07
K=1500 | Precision: 0.01% | Recall: 4.55% | Expected Conversions: 0.13
K=2000 | Precision: 0.02% | Recall: 12.12% | Expected Conversions: 0.33
K=2500 | Precision: 0.02% | Recall: 25.76% | Expected Conversions: 0.47

y_score='altpay1_turnover' ASC
K=500  | Precision: 0.00% | Recall: 0.00% | Expected Conversions: 0.00
K=1000 | Precision: 0.01% | Recall: 1.52% | Expected Conversions: 0.07
K=1500 | Precision: 0.01% | Recall: 6.06% | Expected Conversions: 0.13
K=2000 | Precision: 0.01% | Recall: 12.12% | Expected Conversions: 0.27
K=2500 | Precision: 0.01% | Recall: 14.39% | Expected Conversions: 0.33

y_score='altpay2_turnover' ASC
K=500  | Precision: 0.00% | Recall: 0.00% | Expected Conversions: 0.00
K=1000 | Precision: 0.02% | Recall: 7.58% | Expected Conversions: 0.20
K=1500 | Prec

## Altpay5

### Случайный отбор

In [37]:
k_values = [500, 1000, 1500, 2000, 2500]

target_monthly = df_altpay5.groupby('month').target.sum()
month_sizes = df_altpay5.groupby('month').size()

precision = (target_monthly / month_sizes).mean()  # macro average

print("Random Selection P1P2")

for k in k_values:
    coverage = (k / month_sizes).mean()
    expected_conversions = precision * k

    print(
        f"K={k:<4} | "
        f"Precision: {precision:.2%} | "
        f"Coverage: {coverage:.2%} | "
        f"Expected Conversions: {expected_conversions:.2f}"
    )

Random Selection P1P2
K=500  | Precision: 0.14% | Coverage: 1.97% | Expected Conversions: 0.68
K=1000 | Precision: 0.14% | Coverage: 3.94% | Expected Conversions: 1.35
K=1500 | Precision: 0.14% | Coverage: 5.91% | Expected Conversions: 2.03
K=2000 | Precision: 0.14% | Coverage: 7.88% | Expected Conversions: 2.70
K=2500 | Precision: 0.14% | Coverage: 9.85% | Expected Conversions: 3.38


### Business-Rule

Критерии релевантности:
- Отсутствие подключённого продукта `Altpay5`
- Статус `'Active'`, `'Reborn'`
- Только релевантные MCC (флаг `is_relevant_mcc_altpay5`)
- Минимальный порог оборота (флаг `is_relevant_turnover_altpay5`)

In [38]:
def add_rule_flag_altpay5(df: pd.DataFrame) -> pd.DataFrame:
    active_statuses = ["Active", "Reborn"]

    df = df.copy()
    
    df["rule_flag"] = (
        df["inn_status"].isin(active_statuses)
        & df["is_relevant_mcc_altpay5"]
        & df["is_relevant_turnover_altpay5"]
    ).astype("int8")

    return df


df_altpay5 = add_rule_flag_altpay5(df_altpay5)

In [39]:
k_values = [500, 1000, 1500, 2000, 2500]
y_score = 'turnover'

print(f"Business-Rule Selection Altpay5, y_score='{y_score}'")

for k in k_values:
    df_k = (
        df_altpay5.query('rule_flag == 1')
        .sort_values(by=y_score, ascending=False)
        .groupby('month')
        .head(k)
    )

    tp = df_k[df_k['target'] == 1].groupby('month').size()
    fp = df_k[df_k['target'] == 0].groupby('month').size()

    total_positives = (
        df_altpay5[df_altpay5['target'] == 1]
        .groupby('month')
        .size()
    )

    tp, fp = tp.align(fp, fill_value=0)
    tp, total_positives = tp.align(total_positives, fill_value=0)

    precision_k = (tp / (tp + fp)).mean()
    recall_k = (tp / total_positives).mean()

    avg_rows = df_k.groupby('month').size().mean()
    expected_conversions = precision_k * avg_rows

    print(
        f"K={k:<4} | "
        f"Precision: {precision_k:.2%} | "
        f"Recall: {recall_k:.2%} | "
        f"Expected Conversions: {expected_conversions:.2f}"
    )

Business-Rule Selection Altpay5, y_score='turnover'
K=500  | Precision: 1.00% | Recall: 16.40% | Expected Conversions: 5.00
K=1000 | Precision: 0.90% | Recall: 27.94% | Expected Conversions: 9.00
K=1500 | Precision: 0.79% | Recall: 36.16% | Expected Conversions: 11.80
K=2000 | Precision: 0.76% | Recall: 45.31% | Expected Conversions: 15.20
K=2500 | Precision: 0.74% | Recall: 51.25% | Expected Conversions: 17.29


Пересмотрим метрику ранжирования

In [ ]:
numeric_features = [col for col in df_altpay5.select_dtypes(include=['number']).columns if col not in ('target', 'rule_flag', 'altpay5_turnover')]

k_values = [500, 1000, 1500, 2000, 2500]

print(f"Business-Rule Selection Altpay5")

for asc in (True, False):
    for feature in numeric_features:
        y_score = feature
        print(f"y_score='{y_score}' {'ASC' if asc else 'DESC'}")
        for k in k_values:
            df_k = (
                df_altpay5.query('rule_flag == 1')
                .sort_values(by=y_score, ascending=asc)
                .groupby('month')
                .head(k)
            )

            tp = df_k[df_k['target'] == 1].groupby('month').size()
            fp = df_k[df_k['target'] == 0].groupby('month').size()

            total_positives = (
                df_altpay5[df_altpay5['target'] == 1]
                .groupby('month')
                .size()
            )

            tp, fp = tp.align(fp, fill_value=0)
            tp, total_positives = tp.align(total_positives, fill_value=0)

            precision_k = (tp / (tp + fp)).mean()
            recall_k = (tp / total_positives).mean()

            avg_rows = df_k.groupby('month').size().mean()
            expected_conversions = precision_k * avg_rows

            print(
                f"K={k:<4} | "
                f"Precision: {precision_k:.2%} | "
                f"Recall: {recall_k:.2%} | "
                f"Expected Conversions: {expected_conversions:.2f}"
            )
        print()

Business-Rule Selection P3
y_score='lifetime_month_streak_inn' ASC
K=500  | Precision: 1.49% | Recall: 21.09% | Expected Conversions: 7.47
K=1000 | Precision: 1.14% | Recall: 34.56% | Expected Conversions: 11.40
K=1500 | Precision: 0.94% | Recall: 42.25% | Expected Conversions: 14.07
K=2000 | Precision: 0.78% | Recall: 46.89% | Expected Conversions: 15.53
K=2500 | Precision: 0.74% | Recall: 51.25% | Expected Conversions: 17.29

y_score='altpay1_turnover' ASC
K=500  | Precision: 0.81% | Recall: 10.79% | Expected Conversions: 4.07
K=1000 | Precision: 0.79% | Recall: 21.75% | Expected Conversions: 7.93
K=1500 | Precision: 0.76% | Recall: 31.58% | Expected Conversions: 11.40
K=2000 | Precision: 0.73% | Recall: 40.64% | Expected Conversions: 14.60
K=2500 | Precision: 0.74% | Recall: 51.25% | Expected Conversions: 17.29

y_score='altpay2_turnover' ASC
K=500  | Precision: 0.69% | Recall: 9.22% | Expected Conversions: 3.47
K=1000 | Precision: 0.60% | Recall: 18.29% | Expected Conversions: 6.00